# Predicting steering angle from a single camera frame (A2D2)

A small end-to-end project that takes the front-center camera and CAN-bus data from the public
[Audi A2D2 dataset](https://www.a2d2.audi), aligns them in time, and trains a ResNet-18 to regress the steering-wheel angle from one image.

**Pipeline**

1. Read the bus signals and reconstruct a signed steering angle (the dataset stores magnitude and sign separately).
2. Sanity-check the sign against the gyro yaw rate.
3. Stream a subsample of camera frames straight from S3 into RAM at low resolution (no files, no DataLoader).
4. Time-align frames with steering (`merge_asof`).
5. Fine-tune a pretrained ResNet-18 on the GPU with mixed precision.
6. Compare against trivial baselines so the result means something.

Designed on a free Colab T4.

In [ ]:
import io, json, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

try:
    import boto3
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "boto3"], check=True)
    import boto3
from botocore import UNSIGNED
from botocore.config import Config

import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

# config
BUCKET = "audi-autonomous-driving-dataset"
SCENE  = "20180810_150607"
CAM    = "cam_front_center"
PREFIX = f"camera_lidar/{SCENE}"

STRIDE = 8              # keep every 8th frame
H, W   = 128, 208       # input res
EPOCHS = 10
BATCH  = 64
SEED   = 0
CACHE  = Path("/content/a2d2_cache.npz")

torch.manual_seed(SEED); np.random.seed(SEED)
device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED, max_pool_connections=64))
print("device:", device)

## 1. Steering signal from the CAN bus

The bus gives steering magnitude and a separate sign flag. Rather than guess which flag value means
"left", I picked the convention whose sign agrees with the gyro yaw rate.

In [ ]:
def read_json(key):
    return json.loads(s3.get_object(Bucket=BUCKET, Key=key)["Body"].read())

bus = read_json(f"{PREFIX}/bus/{SCENE.replace('_', '')}_bus_signals.json")

def signal(name):
    """One bus signal as a (t, value) frame indexed by UTC time."""
    df = pd.DataFrame(bus[name]["values"], columns=["t_us", name])
    df["t"] = pd.to_datetime(df["t_us"].astype("int64"), unit="us", utc=True)
    return df[["t", name]].sort_values("t")

tol = pd.Timedelta("10ms")
bus_df = signal("steering_angle_calculated").rename(columns={"steering_angle_calculated": "steer_mag"})
for name, short in [("steering_angle_calculated_sign", "steer_sign"),
                    ("angular_velocity_omega_z", "yaw")]:
    bus_df = pd.merge_asof(bus_df, signal(name).rename(columns={name: short}),
                           on="t", direction="nearest", tolerance=tol)
bus_df = bus_df.dropna().reset_index(drop=True)

candidate = bus_df["steer_mag"] * np.where(bus_df["steer_sign"] == 1, 1.0, -1.0)
corr = candidate.corr(bus_df["yaw"])
bus_df["steer_deg"] = candidate if corr > 0 else -candidate   # same sign convention as yaw rate
final_corr = bus_df["steer_deg"].corr(bus_df["yaw"])
print(f"corr(steering, yaw rate) = {final_corr:+.3f}   rows = {len(bus_df)}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
bus_df["steer_deg"].hist(bins=100, ax=ax[0])
ax[0].set(title="signed steering angle", xlabel="deg", ylabel="samples")
ax[1].scatter(bus_df["yaw"], bus_df["steer_deg"], s=1, alpha=0.15)
ax[1].set(title=f"sanity check vs yaw rate (r = {final_corr:+.2f})", xlabel="yaw rate", ylabel="steering [deg]")
plt.tight_layout(); plt.show()

## 2. Load a subsample of frames straight into RAM

List the frames on S3, keep every `STRIDE`-th one, and fetch + decode + downscale them in parallel threads.
Nothing is written to disk except one small cache file, so re-running the notebook should be instant.

In [ ]:
def load_frame(png_key):
    raw = s3.get_object(Bucket=BUCKET, Key=png_key)["Body"].read()
    img = Image.open(io.BytesIO(raw)).convert("RGB").resize((W, H), Image.BILINEAR, reducing_gap=2.0)
    meta = read_json(png_key[:-4] + ".json")
    return np.asarray(img, dtype=np.uint8), int(meta["cam_tstamp"])

if CACHE.exists():
    d = np.load(CACHE)
    X, stamps = d["X"], d["stamps"]
    print("loaded cache:", X.shape)
else:
    keys = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=f"{PREFIX}/camera/{CAM}/"):
        keys += [o["Key"] for o in page.get("Contents", []) if o["Key"].endswith(".png")]
    keys = sorted(keys)[::STRIDE]
    print(f"{len(keys)} frames selected")

    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=32) as ex:
        out = list(tqdm(ex.map(load_frame, keys), total=len(keys)))
    X = np.stack([o[0] for o in out])
    stamps = np.array([o[1] for o in out], dtype=np.int64)
    np.savez(CACHE, X=X, stamps=stamps)
    print(f"loaded {X.shape} in {time.perf_counter() - t0:.0f}s")

## 3. Align frames with steering and split

Each frame gets the nearest steering sample (within 50 ms). The split is **chronological** (first 90% train,
last 10% validation): neighbouring frames are almost identical, so a random split would leak and look much
better than it really is.

In [ ]:
frames = pd.DataFrame({"idx": np.arange(len(stamps)),
                       "t": pd.to_datetime(stamps, unit="us", utc=True)}).sort_values("t")
frames = pd.merge_asof(frames, bus_df[["t", "steer_deg"]], on="t",
                       direction="nearest", tolerance=pd.Timedelta("50ms")).dropna()

X = X[frames["idx"].values]
y = frames["steer_deg"].values.astype(np.float32)
t = frames["t"].values

n_tr = int(0.9 * len(y))
SCALE = float(y[:n_tr].std())          # targets are trained in units of 1 std

# Baselines the model has to compete against
mae_zero   = np.abs(y[n_tr:] - 0).mean()
mae_median = np.abs(y[n_tr:] - np.median(y[:n_tr])).mean()
print(f"train {n_tr}  val {len(y) - n_tr}   target std {SCALE:.1f} deg")
print(f"baseline val MAE  'always straight': {mae_zero:.1f} deg   'train median': {mae_median:.1f} deg")

# Everything lives on the GPU as uint8
to_gpu = lambda a: torch.from_numpy(a).permute(0, 3, 1, 2).contiguous().to(device)
Xtr, Xva = to_gpu(X[:n_tr]), to_gpu(X[n_tr:])
ytr = torch.tensor(y[:n_tr] / SCALE, device=device)
yva = y[n_tr:]

In [ ]:
idx = np.linspace(0, len(y) - 1, 6).astype(int)
fig, axes = plt.subplots(2, 3, figsize=(12, 4.5))
for ax, i in zip(axes.ravel(), idx):
    ax.imshow(X[i]); ax.axis("off"); ax.set_title(f"steering {y[i]:+.0f}°", fontsize=10)
plt.tight_layout(); plt.show()

## 4. Model and training

ImageNet-pretrained ResNet-18 with a single-output head, Huber loss, AdamW with a one-cycle schedule, and
mixed precision. Augmentation runs on the GPU: brightness/contrast jitter, and a horizontal flip **with the
steering label negated** (a mirrored road scene implies the mirrored steering command).

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def prep(xb, augment=False):
    x = xb.float() / 255
    if augment:
        b = 1 + 0.4 * (torch.rand(len(x), 1, 1, 1, device=device) - 0.5)   # brightness +-20%
        c = 1 + 0.4 * (torch.rand(len(x), 1, 1, 1, device=device) - 0.5)   # contrast   +-20%
        x = (((x - 0.5) * c + 0.5) * b).clamp(0, 1)
    return ((x - MEAN) / STD).contiguous(memory_format=torch.channels_last)

class SteeringNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = resnet18(weights=ResNet18_Weights.DEFAULT)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, 1)
    def forward(self, x):
        return self.backbone(x).squeeze(-1)

model   = SteeringNet().to(device).to(memory_format=torch.channels_last)
loss_fn = nn.SmoothL1Loss()
steps   = int(np.ceil(n_tr / BATCH))
opt     = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched   = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, epochs=EPOCHS, steps_per_epoch=steps)
scaler  = torch.amp.GradScaler("cuda", enabled=use_amp)

@torch.no_grad()
def predict(Xs):
    model.eval()
    out = []
    for i in range(0, len(Xs), 256):
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            out.append(model(prep(Xs[i:i + 256])).float().cpu())
    return torch.cat(out).numpy() * SCALE

history = []
t0 = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(n_tr, device=device)
    run = 0.0
    for i in range(0, n_tr, BATCH):
        b = perm[i:i + BATCH]
        xb, yb = Xtr[b], ytr[b]
        flip = torch.rand(len(b), device=device) < 0.5                      # mirror + negate label
        xb = torch.where(flip.view(-1, 1, 1, 1), xb.flip(3), xb)
        yb = torch.where(flip, -yb, yb)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            loss = loss_fn(model(prep(xb, augment=True)).float(), yb)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
        run += loss.item() * len(b)
    val_mae = np.abs(predict(Xva) - yva).mean()
    history.append((run / n_tr, val_mae))
    print(f"epoch {epoch:2d} | train loss {run / n_tr:.4f} | val MAE {val_mae:6.2f} deg | {time.perf_counter() - t0:4.0f}s")

## 5. Results

In [ ]:
pred = predict(Xva)
mae  = np.abs(pred - yva).mean()
r    = np.corrcoef(pred, yva)[0, 1]

print(f"{'model':<22}{'val MAE [deg]':>14}")
print(f"{'always straight':<22}{mae_zero:>14.1f}")
print(f"{'train median':<22}{mae_median:>14.1f}")
print(f"{'ResNet-18 (ours)':<22}{mae:>14.1f}      (r = {r:+.2f})")

fig, ax = plt.subplots(1, 3, figsize=(17, 3.8))
ax[0].plot([h[0] for h in history], label="train loss")
ax[0].set(title="training loss", xlabel="epoch"); ax[0].legend()
ax[1].plot(t[n_tr:], yva, lw=1, label="true")
ax[1].plot(t[n_tr:], pred, lw=1, label="predicted")
ax[1].set(title="validation drive", ylabel="steering [deg]"); ax[1].legend()
ax[2].scatter(yva, pred, s=6, alpha=0.4)
lim = [min(yva.min(), pred.min()), max(yva.max(), pred.max())]
ax[2].plot(lim, lim, "k--", lw=1)
ax[2].set(title="predicted vs true", xlabel="true [deg]", ylabel="predicted [deg]")
plt.tight_layout(); plt.show()

torch.save(model.state_dict(), "steering_resnet18.pt")

## Limitations and next steps

- **One scene, one camera.** Validation is the last ~10% of a single drive, so results say little about generalisation to new roads.
- **Single frame, no speed.** Steering depends on speed and recent motion; feeding the previous frame or vehicle speed is the obvious next step.
- **Heavy-tailed target.** Most samples are near 0°; large angles (parking, tight turns) are rare, so MAE is dominated by the easy straight driving. Compare against the baselines above rather than reading the number alone.
- Easy extensions: more scenes from the `camera_lidar` subset, a crop of the road region instead of a full-frame resize, a small temporal model.